In [ ]:
use_gpu = True
if use_gpu:
    import tensorly as tl
    tl.set_backend('pytorch')

In [ ]:
import sys
import scipy
import scanpy
import matplotlib
import liana as li
import cell2cell as c2c
import liana as li
import matplotlib.pyplot as plt
import sys
import scanpy as sc
import numpy as np
import pandas as pd
from liana.mt import rank_aggregate
import re
import plotnine as p9
import anndata as ad
import pickle 
import decoupler as dc
import omnipath as op
import pydeseq2
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

# Load Data

In [ ]:
# ---- CONFIG -----------------------------------------------------------------
# Edit ANALYSIS_DIR to match your environment (or set it in the environment).
ANALYSIS_DIR = os.environ.get("ANALYSIS_DIR", ".")
OBJECTS_DIR = os.path.join(ANALYSIS_DIR, "seurat_objects")
 
# Written by the LIANA export R script
F_CAR_MYELOID = "CAR_myeloid_TDN_D0_LIANA.h5ad"
F_NONCAR_MYELOID = "NonCAR_myeloid_TDN_D0_LIANA.h5ad"
F_COMBINED = "full_dataset_for_LIANA_processed.h5ad"
 
# T/NK cell types that get a "_CAR" suffix so CAR+ and CAR- cells of the same
# type stay distinguishable after the two objects are concatenated.
SUFFIX = "_CAR"
TNK_CELLTYPES = [
    "CD4 TEM-like", "CD4 Naive", "CD8 TEM", "CD8 Naive",
    "MAIT", "NK", "Proliferating", "Treg", "gdT",
]
 
# ---- Load -------------------------------------------------------------------
car_dataset = sc.read_h5ad(os.path.join(OBJECTS_DIR, F_CAR_MYELOID))
noncar_dataset = sc.read_h5ad(os.path.join(OBJECTS_DIR, F_NONCAR_MYELOID))
 
# ---- Rename CAR cells -------------------------------------------------------
# cast out of categorical first, otherwise the suffixed labels are not valid categories
car_dataset.obs["cell.anno"] = car_dataset.obs["cell.anno"].astype(str)
 
mask = car_dataset.obs["cell.anno"].isin(TNK_CELLTYPES)
car_dataset.obs.loc[mask, "cell.anno"] = car_dataset.obs.loc[mask, "cell.anno"] + SUFFIX
print(car_dataset.obs["cell.anno"].unique())
 
# ---- Keep only the CAR+ T/NK cells from the CAR object ----------------------
celltypes_keep = [ct + SUFFIX for ct in TNK_CELLTYPES]
car_dataset_sub = car_dataset[car_dataset.obs["cell.anno"].isin(celltypes_keep)].copy()
 
# ---- Combine ----------------------------------------------------------------
# The myeloid cells come from the non-CAR object only, so they are not duplicated.
full_dataset = ad.concat(
    [car_dataset_sub, noncar_dataset],
    axis=0,           
    join="inner",     
    label="batch",    
    keys=["CAR", "NonCAR"],
)
 
print(full_dataset.obs["cell.anno"].unique())
 
full_dataset.write(os.path.join(OBJECTS_DIR, F_COMBINED))


### Optional: explore custom databases 

In [ ]:
resource = li.rs.select_resource('consensus')
resource.head()

lr_pairs = li.resource.select_resource('consensus')

# Run LIANA, TDN-D0 Combined

In [ ]:
ANALYSIS_DIR = os.environ.get("ANALYSIS_DIR", ".")
CSV_DIR = os.path.join(ANALYSIS_DIR, "python_data", "LIANA_CSVs")
os.makedirs(CSV_DIR, exist_ok=True)
 
# Shared rank_aggregate settings, so the three runs cannot drift apart
SAMPLE_KEY = "sample_timepoint2"
GROUPBY = "cell.anno"        # canonical cell-type column
EXPR_PROP = 0.1
MIN_CELLS = 29
N_PERMS = 100
KEY_ADDED = "liana_res"
 
F_CAR_RES = "CAR_myeloid_NK_LIANA_bysample_TDN_D0.csv"
F_NONCAR_RES = "NonCAR_myeloid_NK_LIANA_bysample_TDN_D0.csv"
F_FULL_RES = "FullDataset_LIANA_bysample_TDN_D0.csv"
 

In [ ]:
# ---- Build the grouping columns ---------------------------------------------
# .obs columns come back categorical after an .h5ad round-trip, so cast to str
# before concatenating -- categorical + str raises.
for adata in (car_dataset, noncar_dataset, full_dataset):
    adata.obs["response"] = adata.obs["response"].astype(str)
    adata.obs["timepoint2"] = adata.obs["timepoint2"].astype(str)
    adata.obs["patient.id"] = adata.obs["patient.id"].astype(str)
 
    adata.obs["response_timepoint"] = adata.obs["response"] + "_" + adata.obs["timepoint2"]
    adata.obs["sample_timepoint2"] = adata.obs["patient.id"] + "_" + adata.obs["timepoint2"]
 

In [ ]:
# check group sizes 
for name, adata in (("CAR", car_dataset),
                    ("NonCAR", noncar_dataset),
                    ("Full", full_dataset)):
    print(name, "response_timepoint:", Counter(adata.obs["response_timepoint"]))
    print(name, "sample_timepoint2:", Counter(adata.obs["sample_timepoint2"]))

In [ ]:
# ---- CAR --------------------------------------------------------------------
li.mt.rank_aggregate.by_sample(car_dataset,
                         sample_key = SAMPLE_KEY,
                         groupby = GROUPBY,
                         use_raw = False,
                         resource_name = 'consensus',
                         expr_prop = EXPR_PROP,
                         verbose = True,
                         min_cells = MIN_CELLS,
                         n_perms = N_PERMS,
                         inplace = True,
                         return_all_lrs = True,   # return all LR values
                         key_added = KEY_ADDED)
 
car_dataset.uns[KEY_ADDED].to_csv(os.path.join(CSV_DIR, F_CAR_RES), index=False)
 

In [ ]:
# ---- non-CAR ----------------------------------------------------------------
li.mt.rank_aggregate.by_sample(noncar_dataset,
                         sample_key = SAMPLE_KEY,
                         groupby = GROUPBY,
                         use_raw = False,
                         resource_name = 'consensus',
                         expr_prop = EXPR_PROP,
                         verbose = True,
                         min_cells = MIN_CELLS,
                         n_perms = N_PERMS,
                         inplace = True,
                         return_all_lrs = True,
                         key_added = KEY_ADDED)
 
noncar_dataset.uns[KEY_ADDED].to_csv(os.path.join(CSV_DIR, F_NONCAR_RES), index=False)
 

In [ ]:
 
# ---- Full dataset (CAR + non-CAR combined) ----------------------------------
li.mt.rank_aggregate.by_sample(full_dataset,
                         sample_key = SAMPLE_KEY,
                         groupby = GROUPBY,
                         use_raw = False,
                         resource_name = 'consensus',
                         expr_prop = EXPR_PROP,
                         verbose = True,
                         min_cells = MIN_CELLS,
                         n_perms = N_PERMS,
                         inplace = True,
                         return_all_lrs = True,
                         key_added = KEY_ADDED)
 
full_dataset.uns[KEY_ADDED].to_csv(os.path.join(CSV_DIR, F_FULL_RES), index=False)


# DEA analysis for CAR+ and CAR- 

In [ ]:
# ---- CONFIG -----------------------------------------------------------------
ANALYSIS_DIR = os.environ.get("ANALYSIS_DIR", ".")
DEA_DIR = os.path.join(ANALYSIS_DIR, "python_data", "DEA_save")
LR_DIR = os.path.join(ANALYSIS_DIR, "python_data", "LIANA_CSVs")
for d in (DEA_DIR, LR_DIR):
    os.makedirs(d, exist_ok=True)

GROUP_BY = "cell.anno"          # canonical cell-type column
SAMPLE_COL = "sample.name"
CONDITION_KEY = "response"
REF_LEVEL = "PD"                # control / PD
TEST_LEVEL = "CR"               # CR positive 

# True runs pyDESeq2 from scratch and saves the tables to DEA_DIR;
# set to False on later runs to reuse those saved tables.
RECOMPUTE_DEA = True

CELL_GROUPS = [
    "CD14 Mono", "CD16 Mono", "CD4 TEM-like", "CD4 Naive", "CD8 TEM",
    "CD8 Naive", "Dendritic Cell", "NK", "Proliferating", "Treg", "gdT",
]

In [ ]:
# ---- Add counts layer -------------------------------------------------------
car_dataset.layers["counts"] = car_dataset.raw.X
noncar_dataset.layers["counts"] = noncar_dataset.raw.X

In [ ]:
# ---- Pseudobulk by sample x cell type ---------------------------------------
car_pdata_sample = dc.pp.pseudobulk(
    car_dataset,
    sample_col=SAMPLE_COL,
    groups_col=GROUP_BY,
    layer="counts",
    mode="sum",
)

noncar_pdata_sample = dc.pp.pseudobulk(
    noncar_dataset,
    sample_col=SAMPLE_COL,
    groups_col=GROUP_BY,
    layer="counts",
    mode="sum",
)

In [ ]:
# filter samples based on number of cells and counts
dc.pp.filter_samples(car_pdata_sample, min_cells=1, min_counts=100)
dc.pp.filter_samples(noncar_pdata_sample, min_cells=1, min_counts=100)

dc.pl.filter_samples(car_pdata_sample, groupby=["response_timepoint", GROUP_BY], figsize=(4, 8))
dc.pl.filter_samples(noncar_pdata_sample, groupby=["response_timepoint", GROUP_BY], figsize=(4, 8))

In [ ]:
# Helper function 
def get_dea_results(subset_obj):
    dea_results = {}
    quiet = False
    for cell_group in CELL_GROUPS:
        print(cell_group)
        # Select cell profiles
        ctdata = subset_obj[subset_obj.obs[GROUP_BY] == cell_group].copy()
        print(ctdata)

        # Sum counts per gene across all cells
        gene_counts = np.array(ctdata.X.sum(axis=0)).flatten()  # flatten in case it's a 2D matrix
        genes = ctdata.var_names[gene_counts >= 1]  # keep genes with at least 1 count
        ctdata = ctdata[:, genes].copy()

        if len(np.unique(ctdata.obs[CONDITION_KEY])) == 1:
            continue

        # Build DESeq2 object
        dds = DeseqDataSet(
            adata=ctdata,
            design_factors=CONDITION_KEY,
            ref_level=[CONDITION_KEY, REF_LEVEL],  # set control as reference
            refit_cooks=True,
            quiet=quiet,
        )

        # Compute LFCs
        dds.deseq2()
        # Contrast between responder and non-responder
        stat_res = DeseqStats(dds, contrast=[CONDITION_KEY, TEST_LEVEL, REF_LEVEL], quiet=quiet)
        stat_res.quiet = quiet
        # Compute Wald test
        stat_res.summary()
        # Shrink LFC using the actual coefficient
        coeff_name = stat_res.LFC.columns[0]  
        stat_res.lfc_shrink(coeff=coeff_name)  # {CONDITION_KEY}_cond_vs_ref

        dea_results[cell_group] = stat_res.results_df
    return dea_results

In [ ]:
# ---- Subsets: pseudobulk and single-cell, per condition ---------------------
# Each entry pairs one pseudobulk subset with the matching single-cell object.
conditions = ["CAR_Peak", "CAR_Week4", "NonCAR_APH", "NonCAR_TDN_D0", "CAR_TDN_D0"]

subset_list = [
    car_pdata_sample[car_pdata_sample.obs["timepoint"] == "Peak"].copy(),
    car_pdata_sample[car_pdata_sample.obs["timepoint"] == "4W"].copy(),
    noncar_pdata_sample[noncar_pdata_sample.obs["timepoint"] == "APH"].copy(),
    noncar_pdata_sample[noncar_pdata_sample.obs["timepoint2"] == "TDN-D0"].copy(),
    car_pdata_sample[car_pdata_sample.obs["timepoint2"] == "TDN-D0"].copy(),
]

adata_list = [
    car_dataset[car_dataset.obs["timepoint"] == "Peak"].copy(),
    car_dataset[car_dataset.obs["timepoint"] == "4W"].copy(),
    noncar_dataset[noncar_dataset.obs["timepoint"] == "APH"].copy(),
    noncar_dataset[noncar_dataset.obs["timepoint2"] == "TDN-D0"].copy(),
    car_dataset[car_dataset.obs["timepoint2"] == "TDN-D0"].copy(),
]

print(subset_list[4].obs[GROUP_BY].unique())

In [ ]:
# RUN DEA -> ligand-receptor ranking, per condition --------------------------
for cond, pdata, adata in zip(conditions, subset_list, adata_list):
    print(cond)

    dea_path = os.path.join(DEA_DIR, f"LIANA_DEA_deseq_DF_{cond}.csv")

    if RECOMPUTE_DEA:
        dea_results = get_dea_results(pdata)
        dea_df = pd.concat(dea_results)
        dea_df = (dea_df
                  .reset_index()
                  .rename(columns={"level_0": GROUP_BY, "level_1": "index"})
                  .set_index("index"))
    else:
        dea_df = pd.read_csv(dea_path).set_index("index")

    # pyDESeq2 sometimes introduces NaN p-values; check before filling
    print("rows with NA:", len(dea_df[dea_df.isna().any(axis=1)]))
    dea_df_filled = dea_df.fillna(1)

    if RECOMPUTE_DEA:
        dea_df_filled.to_csv(dea_path)

    lr_res = li.multi.df_to_lr(
        adata,
        dea_df=dea_df_filled,
        resource_name="consensus",  # NOTE: uses HUMAN gene symbols!
        expr_prop=0.01,             # calculated for adata as passed - used to filter interactions
        groupby=GROUP_BY,
        stat_keys=["stat", "pvalue", "padj"],
        use_raw=False,
        complex_col="stat",         # NOTE: we use the Wald Stat to deal with complexes
        verbose=True,
        return_all_lrs=False,
    )

    lr_res = lr_res.sort_values("interaction_stat", ascending=False, key=abs)
    lr_res.to_csv(os.path.join(LR_DIR, f"LIANA_DEA_{cond}_rank_agg_ALL.csv"), index=True)

# Tensor on full dataset

In [ ]:
# ---- CONFIG -----------------------------------------------------------------
ANALYSIS_DIR = os.environ.get("ANALYSIS_DIR", ".")
OBJECTS_DIR = os.path.join(ANALYSIS_DIR, "python_objects")
FIG_DIR = os.path.join(ANALYSIS_DIR, "python_data", "images", "Tensor",
                       "FullDataset_TDN_D0_bysample")
 
for d in (OBJECTS_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

In [ ]:
SAMPLE_KEY = "sample_timepoint2"
CONTEXT_KEY = "response_timepoint"
SCORE_KEY = "magnitude_rank"
 
F_TENSOR = os.path.join(OBJECTS_DIR, "fulldataset_tensor_bysample_TDN_D0.pkl")
F_TENSOR_META = os.path.join(OBJECTS_DIR, "fulldataset_tensor_bysample_TDN_D0_metadata.pkl")
F_TENSOR_FACTORIZED = os.path.join(OBJECTS_DIR, "fulldataset_tensor_bysample_TDN_D0_factorized.pkl")

In [ ]:
# ---- Build the tensor -------------------------------------------------------
sorted_samples = sorted(full_dataset.uns["liana_res"][SAMPLE_KEY].unique())
 
full_tensor_all = li.multi.to_tensor_c2c(liana_res=full_dataset.uns['liana_res'], # LIANA's dataframe containing results
                                sample_key=SAMPLE_KEY, 
                                source_key='source', # Column name of the sender cells
                                target_key='target', # Column name of the receiver cells
                                ligand_key='ligand_complex', # Column name of the ligands
                                receptor_key='receptor_complex', # Column name of the receptors
                                score_key=SCORE_KEY, # Column name of the communication scores to use
                                inverse_fun=lambda x: 1 - x, 
                                how='outer', # What to include across all samples
                                outer_fraction=0.1, # Fraction of samples as threshold to include cells and LR pairs.
                                context_order=sorted_samples, # Order to store the contexts in the tensor
                                                             )
 
# ---- Tensor metadata --------------------------------------------------------
full_dataset.obs[CONTEXT_KEY] = (full_dataset.obs["response"].astype(str) + "_"
                                 + full_dataset.obs["timepoint2"].astype(str))
 
context_dict = full_dataset.obs.sort_values(by=SAMPLE_KEY) \
                        .set_index(SAMPLE_KEY)[CONTEXT_KEY] \
                        .to_dict()
 
dimensions_dict = [context_dict, None, None, None]
meta_tensor_full = c2c.tensor.generate_tensor_metadata(interaction_tensor=full_tensor_all,
                                                  metadata_dicts=dimensions_dict,
                                                  fill_with_order_elements=True
                                                 )
 
# save
c2c.io.export_variable_with_pickle(variable=full_tensor_all, filename=F_TENSOR)
c2c.io.export_variable_with_pickle(variable=meta_tensor_full, filename=F_TENSOR_META)
 

In [ ]:
# ---- Tensor factorization ---------------------------------------------------
full_tensor_factorized = c2c.analysis.run_tensor_cell2cell_pipeline(full_tensor_all,
                                           meta_tensor_full,
                                           copy_tensor=True,
                                           rank=None, # Number of factors to perform the factorization. If None, it is automatically determined by an elbow analysis
                                           tf_optimization='robust', # To define how robust we want the analysis to be.
                                           random_state=0, # Random seed for reproducibility
                                           upper_rank=25,
                                           device='cuda', # Device to use. If using GPU and PyTorch, use 'cuda'. For CPU use 'cpu'
                                           output_folder=FIG_DIR, # Whether to save the figures in files. If so, a folder pathname must be passed
                                          )
 
c2c.io.export_variable_with_pickle(variable=full_tensor_factorized, filename=F_TENSOR_FACTORIZED)
 
# ---- Resume from disk (only needed in a fresh session) ----------------------
full_tensor_all = c2c.io.load_variable_with_pickle(filename=F_TENSOR)
meta_tensor_full = c2c.io.load_variable_with_pickle(filename=F_TENSOR_META)
 
print(full_tensor_all.shape)
print(full_tensor_all.missing_fraction())
print(full_tensor_all.sparsity_fraction())
print(full_tensor_all.excluded_value_fraction())

# Downstream Analysis

In [ ]:
# ---- CONFIG -----------------------------------------------------------------
ANALYSIS_DIR = os.environ.get("ANALYSIS_DIR", ".")
OBJECTS_DIR = os.path.join(ANALYSIS_DIR, "python_objects")
TENSOR_DIR = os.path.join(ANALYSIS_DIR, "python_data", "images", "Tensor",
                          "FullDataset_TDN_D0_bysample")
 
os.makedirs(TENSOR_DIR, exist_ok=True)
 
SAMPLE_KEY = "sample_timepoint2"
CONTEXT_KEY = "response_timepoint"
 
F_TENSOR_FACTORIZED = os.path.join(OBJECTS_DIR, "fulldataset_tensor_bysample_TDN_D0_factorized.pkl")
F_TENSOR_META = os.path.join(OBJECTS_DIR, "fulldataset_tensor_bysample_TDN_D0_metadata.pkl")
 
# response_timepoint levels, in plotting order
CONTEXT_ORDER = ["CR_APH", "PD_APH",
                 "CR_TDN-D0", "PD_TDN-D0",
                 "CR_Peak", "PD_Peak",
                 "CR_4W", "PD_4W"]
 
TOP_N = 20
LOADING_THRESHOLD = 0.1
 

In [ ]:
# ---- Rebuild the grouping columns -------------------------------------------
full_dataset.obs["patient.id"] = full_dataset.obs["patient.id"].astype(str)
full_dataset.obs["timepoint2"] = full_dataset.obs["timepoint2"].astype(str)
full_dataset.obs["response"] = full_dataset.obs["response"].astype(str)
 
full_dataset.obs[SAMPLE_KEY] = full_dataset.obs["patient.id"] + "_" + full_dataset.obs["timepoint2"]
full_dataset.obs[CONTEXT_KEY] = full_dataset.obs["response"] + "_" + full_dataset.obs["timepoint2"]
 
print(Counter(full_dataset.obs[SAMPLE_KEY]))
 
context_dict = full_dataset.obs.sort_values(by=SAMPLE_KEY) \
                       .set_index(SAMPLE_KEY)[CONTEXT_KEY] \
                       .to_dict()
dimensions_dict = [context_dict, None, None, None]

In [ ]:
# ---- Load the factorized tensor and its loadings ----------------------------
full_tensor_factorized = c2c.io.load_variable_with_pickle(F_TENSOR_FACTORIZED)
meta_full_tensor = c2c.io.load_variable_with_pickle(F_TENSOR_META)
 
factors = c2c.io.load_tensor_factors(os.path.join(TENSOR_DIR, "Loadings.xlsx"))

In [ ]:
 # ----  (REDONE IN R) Context boxplots with Mann-Whitney tests -------------------------------
_ = c2c.plotting.context_boxplot(context_loadings=full_tensor_factorized.factors['Contexts'],
                                 metadict=context_dict,
                                 nrows=2,
                                 figsize=(10, 20),
                                 statistical_test='Mann-Whitney',
                                 pval_correction=None,
                                 text_format="simple",
                                 cmap='plasma',
                                 verbose=False,
                                 # group_order=CONTEXT_ORDER,
                                 filename=os.path.join(TENSOR_DIR, "Factor_Boxplots_Wilcoxon.pdf")
                                )
 

In [ ]:
# ---- Top ligand-receptor pairs per factor -----------------------------------
lr_loadings = factors['Ligand-Receptor Pairs']
lr_loadings.to_csv(os.path.join(TENSOR_DIR, "LR_loading_values.csv"))
 
for fn in range(1, lr_loadings.shape[1] + 1):
    factor = "Factor " + str(fn)
    print(lr_loadings.sort_values(factor, ascending=False).head(TOP_N))
    
## all factors: keep the top LRs of each
rows_to_keep = []
for col in lr_loadings.columns:
    top_rows = lr_loadings[lr_loadings[col] > LOADING_THRESHOLD].nlargest(TOP_N, col)
    rows_to_keep.extend(top_rows.index.tolist())
    
## remove duplicates while preserving order
rows_to_keep = list(dict.fromkeys(rows_to_keep))
df_filtered = lr_loadings.loc[rows_to_keep]
print(df_filtered.shape)
 
lr_cm = c2c.plotting.loading_clustermap(df_filtered,
                                        loading_threshold=LOADING_THRESHOLD, # To consider only top LRs
                                        figsize=(28, 8),
                                        filename=os.path.join(TENSOR_DIR, "Clustermap-LRs_filtered.pdf"),
                                        row_cluster=False # To avoid clustering of factors
                                       )

In [ ]:
 # selected factors only for more focused results 
cols = ["Factor 2", "Factor 6"]
 
rows_to_keep = []
for col in cols:
    top_rows = lr_loadings[lr_loadings[col] > LOADING_THRESHOLD].nlargest(TOP_N, col)
    rows_to_keep.extend(top_rows.index.tolist())

rows_to_keep = list(dict.fromkeys(rows_to_keep))
df_filtered_2 = lr_loadings.loc[rows_to_keep]
print(df_filtered_2.shape)
 
lr_cm = c2c.plotting.loading_clustermap(df_filtered_2,
                                        loading_threshold=LOADING_THRESHOLD,
                                        use_zscore=False,
                                        figsize=(28, 8),
                                        filename=os.path.join(TENSOR_DIR, "Clustermap-LRs_top20_factor2-6-only.pdf"),
                                        row_cluster=False
                                       )
 
# unfiltered
lr_cm = c2c.plotting.loading_clustermap(lr_loadings,
                                        loading_threshold=LOADING_THRESHOLD,
                                        use_zscore=False,
                                        figsize=(28, 8),
                                        filename=os.path.join(TENSOR_DIR, "Clustermap-LRs.pdf"),
                                        row_cluster=False
                                       )

In [ ]:
 # ---- Sender x receiver joint loadings, per factor ---------------------------
for selected_factor in factors["Contexts"].columns:
    print(selected_factor)
    loading_product = c2c.analysis.tensor_downstream.get_joint_loadings(factors,
                                                                    dim1='Sender Cells',
                                                                    dim2='Receiver Cells',
                                                                    factor=selected_factor
                                                                   )
 
    lprod_cm = c2c.plotting.loading_clustermap(loading_product.T, # Remove .T to transpose the axes
                                           use_zscore=False, # Whether standardizing the loadings across factors
                                           figsize=(8, 8),
                                           filename=os.path.join(TENSOR_DIR, selected_factor + '_Clustermap-CC-Pairs.pdf'),
                                           cbar_label='Loading Product',
                                          )

In [ ]:
 # ---- Factor-specific communication networks ---------------------------------
# Get all outer products as adjacency matrices, one per factor
networks = c2c.analysis.tensor_downstream.get_factor_specific_ccc_networks(factors,
                                                                           sender_label='Sender Cells',
                                                                           receiver_label='Receiver Cells',
                                                                           )
# Then, flatten the adjacency matrices
network_by_factors = c2c.analysis.tensor_downstream.flatten_factor_ccc_networks(networks, orderby='receivers')
 
# And we can plot the distributions of the weights for each factor-specific network
plt.figure(figsize=(8, 6))
plt.hist(network_by_factors.values.flatten(), bins=50, color='skyblue', edgecolor='black')
plt.title("Distribution of factor-specific network weights")
plt.xlabel("Edge weight")
plt.ylabel("Frequency")
plt.tight_layout()
 
c2c.plotting.ccc_networks_plot(factors,
                               included_factors=factors["Contexts"].columns,
                               ccc_threshold=0.05, # Only important communication
                               nrows=1,
                               panel_size=(16, 16), # This changes the size of each figure panel.
                               filename=os.path.join(TENSOR_DIR, "Factor-Networks.pdf")
                              )

In [ ]:
# ---- Gini coefficients ------------------------------------------------------
gini_df = c2c.analysis.tensor_downstream.compute_gini_coefficients(factors,
                                                         sender_label='Sender Cells',
                                                         receiver_label='Receiver Cells'
                                                        )
 
gini_df.to_csv(os.path.join(TENSOR_DIR, "gini_df.csv"))
print(gini_df)
 